<a id="rlc-ode"></a>

$$
\text{RLC 串联电路 (Series RLC Circuit): 电压源 } V_s \text{ 在 t = 0 接入, 初始 } v_C(0) = 0, \ i(0) = 0 \\ \quad \\
\text{1. KVL: } v_L + v_R + v_C = V_s \text{, 元件方程 } v_L = L \frac{di}{dt}, \ v_R = R i, \ i = C \frac{dv_C}{dt} \\
\text{把 } i = C v_C' \text{ 代入, } \frac{di}{dt} = C v_C'' \text{:} \\
L C v_C'' + R C v_C' + v_C = V_s \\
\text{两边除以 LC, 定义 } \omega_0 = \frac{1}{\sqrt{LC}} \text{ (固有角频率 Natural Frequency), } \zeta = \frac{R}{2} \sqrt{\frac{C}{L}} \text{ (阻尼比 Damping Ratio):} \\
\frac{R}{L} = 2 \cdot \frac{R}{2} \sqrt{\frac{C}{L}} \cdot \frac{1}{\sqrt{LC}} = 2 \zeta \omega_0 \\
v_C'' + 2\zeta\omega_0 v_C' + \omega_0^2 v_C = \omega_0^2 V_s \quad (\text{二阶常系数线性 ODE, 和弹簧-质量-阻尼器完全相同}) \\ \quad \\
\text{对应关系: L ↔ 质量 m (惯性), R ↔ 阻尼 c, } \frac{1}{C} \text{ ↔ 弹簧刚度 k, 电荷 q ↔ 位移 x}
$$

数学基础: [高阶 ODE 与特征方程](../math_basics/calculus.ipynb#ode-system) · [欧拉公式](../math_basics/calculus.ipynb#taylor)

<a id="damping"></a>

$$
\text{2. 齐次方程的通解: 猜 } v = e^{rt} \text{, 代入 } v'' + 2\zeta\omega_0 v' + \omega_0^2 v = 0 \\
r^2 + 2\zeta\omega_0 r + \omega_0^2 = 0 \implies r = \frac{-2\zeta\omega_0 \pm \sqrt{4\zeta^2\omega_0^2 - 4\omega_0^2}}{2} = -\zeta\omega_0 \pm \omega_0 \sqrt{\zeta^2 - 1} \\
\text{由判别式 (Discriminant) 的符号分三种情况:} \\ \quad \\
\text{(a) 过阻尼 (Overdamped) } \zeta > 1 \text{: 两个不同的负实根 } r_1, r_2 \text{, } v_h = A e^{r_1 t} + B e^{r_2 t} \text{ (缓慢趋于稳态, 无振荡)} \\
\text{(b) 临界阻尼 (Critically Damped) } \zeta = 1 \text{: 重根 } r = -\omega_0 \text{, } v_h = (A + B t) e^{-\omega_0 t} \\
\quad \text{为什么多了 } t e^{rt} \text{: 代入 } (t e^{rt})'' + 2\zeta\omega_0 (t e^{rt})' + \omega_0^2 t e^{rt} = t e^{rt} \underbrace{(r^2 + 2\zeta\omega_0 r + \omega_0^2)}_{= 0} + e^{rt} \underbrace{(2r + 2\zeta\omega_0)}_{= 0 \text{ (重根 } r = -\zeta\omega_0)} = 0 \\
\text{(c) 欠阻尼 (Underdamped) } \zeta < 1 \text{: 共轭复根 } r = -\zeta\omega_0 \pm j \omega_d, \ \omega_d = \omega_0 \sqrt{1 - \zeta^2} \text{ (阻尼振荡频率)} \\
\quad e^{(-\zeta\omega_0 \pm j\omega_d) t} = e^{-\zeta\omega_0 t} (\cos\omega_d t \pm j \sin\omega_d t) \implies v_h = e^{-\zeta\omega_0 t}(A \cos\omega_d t + B \sin\omega_d t) \\ \quad \\
\text{3. 特解: 右边是常数, 试 } v_p = K \text{: } \omega_0^2 K = \omega_0^2 V_s \implies v_p = V_s \text{ (稳态时电容充满, 电流为 0)} \\ \quad \\
\text{4. 用初始条件 } v(0) = 0, \ v'(0) = \frac{i(0)}{C} = 0 \text{ 确定常数 (以欠阻尼为例):} \\
v(0) = V_s + A = 0 \implies A = -V_s \\
v'(0) = -\zeta\omega_0 A + \omega_d B = 0 \implies B = \frac{\zeta\omega_0}{\omega_d} A = -V_s \frac{\zeta\omega_0}{\omega_d} \\
v_C(t) = V_s \left[1 - e^{-\zeta\omega_0 t}\left(\cos\omega_d t + \frac{\zeta\omega_0}{\omega_d} \sin\omega_d t\right)\right] \\
\text{过阻尼: } A + B = -V_s, \ r_1 A + r_2 B = 0 \implies v_C(t) = V_s \left[1 + \frac{r_2 e^{r_1 t} - r_1 e^{r_2 t}}{r_1 - r_2}\right] \\
\text{临界阻尼: } A = -V_s, \ -\omega_0 A + B = 0 \implies v_C(t) = V_s \left[1 - (1 + \omega_0 t) e^{-\omega_0 t}\right]
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

L, Cap, Vs = 10e-3, 1e-6, 5.0
w0 = 1 / np.sqrt(L * Cap)
print(f'omega_0 = 1/sqrt(LC) = {w0:.1f} rad/s, f_0 = {w0 / (2 * np.pi):.2f} Hz')

def zeta_of(R):
    return R / 2 * np.sqrt(Cap / L)

def step_response(R, t):
    z = zeta_of(R)
    if np.isclose(z, 1):
        return Vs * (1 - (1 + w0 * t) * np.exp(-w0 * t))
    if z < 1:
        wd = w0 * np.sqrt(1 - z**2)
        return Vs * (1 - np.exp(-z * w0 * t) * (np.cos(wd * t) + z * w0 / wd * np.sin(wd * t)))
    r1 = -z * w0 + w0 * np.sqrt(z**2 - 1)
    r2 = -z * w0 - w0 * np.sqrt(z**2 - 1)
    return Vs * (1 + (r2 * np.exp(r1 * t) - r1 * np.exp(r2 * t)) / (r1 - r2))

def rlc_rhs(R, source):
    # 状态 [v_C, i]: v_C' = i / C, i' = (v_s - R i - v_C) / L  (KVL)
    return lambda t, s: [s[1] / Cap, (source(t) - R * s[1] - s[0]) / L]

t = np.linspace(0, 3e-3, 2000)
plt.figure(figsize=(4, 3))
for R in [40.0, 200.0, 600.0]:
    z = zeta_of(R)
    sol = solve_ivp(rlc_rhs(R, lambda t: Vs), [0, t[-1]], [0.0, 0.0], t_eval=t, rtol=1e-10, atol=1e-12)
    analytic = step_response(R, t)
    roots = np.roots([1, 2 * z * w0, w0**2])
    print(f'R = {R:>5.0f} ohm: zeta = {z:.2f}, 特征根 = {np.round(roots, 1)}, 解析解 == solve_ivp ? {np.allclose(analytic, sol.y[0], atol=1e-6)}')
    plt.plot(t * 1000, analytic, label=f'zeta = {z:.1f}')
plt.axhline(Vs, color='gray', linestyle=':')
plt.xlabel('t (ms)')
plt.ylabel('v_C (V)')
plt.title('RLC step response')
plot_nicely()
plt.show()

<a id="overshoot"></a>

$$
\text{欠阻尼的超调量 (Overshoot)} \\
\text{对欠阻尼解求导 (令 } \alpha = \zeta\omega_0 \text{, 乘积法则, cos 项抵消):} \\
\frac{d}{dt}\left[e^{-\alpha t}\left(\cos\omega_d t + \frac{\alpha}{\omega_d} \sin\omega_d t\right)\right] = e^{-\alpha t}\left[-\alpha\cos - \frac{\alpha^2}{\omega_d}\sin - \omega_d \sin + \alpha\cos\right] = -e^{-\alpha t} \frac{\alpha^2 + \omega_d^2}{\omega_d} \sin\omega_d t \\
\alpha^2 + \omega_d^2 = \zeta^2\omega_0^2 + \omega_0^2(1 - \zeta^2) = \omega_0^2 \implies v_C'(t) = V_s \frac{\omega_0^2}{\omega_d} e^{-\zeta\omega_0 t} \sin\omega_d t \\
\text{第一个峰值: } \sin\omega_d t = 0 \text{ 的第一个正根 } t_p = \frac{\pi}{\omega_d} \text{, 代入 (} \cos\pi = -1, \sin\pi = 0 \text{):} \\
v_C(t_p) = V_s\left(1 + e^{-\zeta\omega_0 \pi / \omega_d}\right) \implies \text{超调量 } M_p = \frac{v_C(t_p) - V_s}{V_s} = e^{-\frac{\zeta\pi}{\sqrt{1 - \zeta^2}}} \\
\text{只和阻尼比有关, } \zeta \to 0 \text{ 时 } M_p \to 100\% \text{ (电容电压可以冲到两倍电源电压)}
$$

数学基础: [乘积法则](../math_basics/calculus.ipynb#derivative-rules) · [极值的一阶条件](../math_basics/calculus.ipynb#extremum)

In [ ]:
for R in [10.0, 40.0, 100.0, 160.0]:
    z = zeta_of(R)
    wd = w0 * np.sqrt(1 - z**2)
    t_fine = np.linspace(0, 2 * np.pi / wd, 20001)
    v = step_response(R, t_fine)
    k = np.argmax(v)
    print(f'zeta = {z:.2f}: 峰值时刻 数值 = {t_fine[k] * 1e6:.2f} us, pi/w_d = {np.pi / wd * 1e6:.2f} us; '
          f'超调 数值 = {(v[k] - Vs) / Vs * 100:.3f}%, 公式 = {np.exp(-z * np.pi / np.sqrt(1 - z**2)) * 100:.3f}%')

<a id="resonance"></a>

$$
\text{谐振 (Resonance): 正弦电压 } v_s = V_m \cos\omega t \text{ 驱动, 稳态电流的振幅随频率的变化} \\ \quad \\
\text{用复指数代替余弦 (} \cos\omega t = \text{Re}(e^{j\omega t}) \text{, 线性方程的实部仍是解): 设 } i = \text{Re}(I e^{j\omega t}) \\
\text{求导只是乘以 } j\omega \text{: } \frac{d}{dt} e^{j\omega t} = j\omega e^{j\omega t}, \ \int e^{j\omega t} dt = \frac{1}{j\omega} e^{j\omega t} \\
\text{KVL 对电流写: } L \frac{di}{dt} + R i + \frac{1}{C}\int i \, dt = v_s \implies \left(j\omega L + R + \frac{1}{j\omega C}\right) I = V_m \\
|I| = \frac{V_m}{\sqrt{R^2 + \left(\omega L - \frac{1}{\omega C}\right)^2}} \quad (\text{详细见 phasor_and_impedance 的阻抗}) \\ \quad \\
\text{1. 谐振频率: 分母最小 } \iff \omega L = \frac{1}{\omega C} \implies \omega_0 = \frac{1}{\sqrt{LC}} \text{, 此时 } |I|_{max} = \frac{V_m}{R} \text{ (电感和电容的电压正好抵消)} \\
\text{2. 带宽 (Bandwidth): 功率降到一半 (半功率点 } |I|^2 = \frac{1}{2}|I|_{max}^2 \text{, 即 -3 dB)} \\
R^2 + \left(\omega L - \frac{1}{\omega C}\right)^2 = 2R^2 \implies \omega L - \frac{1}{\omega C} = \pm R \implies L\omega^2 \mp R\omega - \frac{1}{C} = 0 \\
\text{取正根: } \omega_{1,2} = \mp\frac{R}{2L} + \sqrt{\left(\frac{R}{2L}\right)^2 + \omega_0^2} \implies \Delta\omega = \omega_2 - \omega_1 = \frac{R}{L} \\
\text{3. 品质因数 (Quality Factor): } Q = \frac{\omega_0}{\Delta\omega} = \frac{\omega_0 L}{R} = \frac{1}{R}\sqrt{\frac{L}{C}} = \frac{1}{2\zeta} \\
\text{Q 越大 (R 越小), 谐振峰越尖锐, 选频能力越强 (收音机调台就是调 C 改变 } \omega_0 \text{)}
$$

数学基础: [欧拉公式](../math_basics/calculus.ipynb#taylor) · 应用: [RMS 与平均功率](AC_DC.ipynb#ac-power)

In [ ]:
def current_amplitude(R, w):
    return Vm / np.sqrt(R**2 + (w * L - 1 / (w * Cap))**2)

Vm = 1.0
# 1. 直接解 ODE, 等暂态衰减后测量稳态电流振幅, 和公式对比
R_test = 40.0
for w in [0.5 * w0, 0.9 * w0, w0, 1.1 * w0, 2 * w0]:
    T_sim = 10 / (zeta_of(R_test) * w0) + 5 * 2 * np.pi / w     # 暂态衰减 + 若干周期
    t_eval = np.linspace(T_sim - 3 * 2 * np.pi / w, T_sim, 3000)
    sol = solve_ivp(rlc_rhs(R_test, lambda t: Vm * np.cos(w * t)), [0, T_sim], [0.0, 0.0], t_eval=t_eval,
                    rtol=1e-9, atol=1e-12, max_step=2 * np.pi / w / 50)
    i_ss = sol.y[1]
    print(f'w = {w / w0:.1f} w0: ODE 稳态电流振幅 = {(i_ss.max() - i_ss.min()) / 2 * 1000:.4f} mA, 公式 = {current_amplitude(R_test, w) * 1000:.4f} mA')

# 2. 共振曲线和带宽
w_grid = np.linspace(0.3 * w0, 2.5 * w0, 20000)
plt.figure(figsize=(4, 3))
for R in [10.0, 40.0, 100.0]:
    amp = current_amplitude(R, w_grid)
    half = amp**2 >= 0.5 * amp.max()**2
    bw_numeric = w_grid[half][-1] - w_grid[half][0]
    peak = w_grid[np.argmax(amp)]
    print(f'R = {R:>5.0f}: 峰值频率 = {peak / w0:.4f} w0, 带宽 数值 = {bw_numeric:.1f}, R/L = {R / L:.1f} rad/s, '
          f'Q = w0 L / R = {w0 * L / R:.2f} == 1/(2 zeta) = {1 / (2 * zeta_of(R)):.2f}')
    plt.plot(w_grid / w0, amp * 1000, label=f'R = {R:.0f}, Q = {w0 * L / R:.1f}')
plt.xlabel('omega / omega_0')
plt.ylabel('|I| (mA)')
plot_nicely()
plt.show()